In [1]:
# Standard library
import os  
from pathlib import Path  # Object-oriented filesystem paths

# Data handling packages
import numpy as np  
import numpy.random as npr  
import pandas as pd 
import pynwb
import ipywidgets as widgets
from IPython.display import display
import gc
import time
import sklearn

# Progress bar utility
from tqdm import tqdm  # Displays a smart progress bar during loops

# Preprocessing
from sklearn.preprocessing import StandardScaler  # Standardizes features (zero mean, unit variance)
from sklearn.svm import LinearSVC
from sklearn import svm
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import confusion_matrix

# Plotting libraries
import matplotlib.pyplot as plt  
from matplotlib import colors  
import seaborn as sns  

# HMM-related imports from JAX, Dynamax, and TensorFlow Probability
from functools import partial
import jax.numpy as jnp
import jax.random as jr
from dynamax.hidden_markov_model import GaussianHMM
from dynamax.utils.utils import find_permutation

# Additional HMM variants and plotting utilities from Dynamax
from dynamax.hidden_markov_model import (
    DiagonalGaussianHMM,
    SphericalGaussianHMM,
    SharedCovarianceGaussianHMM
)

from dynamax.utils.plotting import CMAP, COLORS, white_to_color_cmap

# Pandas display settings
pd.set_option('display.max_columns', None)  # Ensures all columns are shown when printing DataFrames

# Inline plotting for Jupyter Notebooks
%matplotlib inline  


In [2]:
root = Path("/root/capsule/data/dynamicrouting_datacube")

example_session_ids = [
                        "759434_2025-02-04", "713655_2024-08-09", "743199_2024-12-05", 
                        "712815_2024-05-22", "742903_2024-10-23", "664851_2023-11-16", 
                        "741137_2024-10-10", "662892_2023-08-24", "714748_2024-06-24",
                        "667252_2023-09-28", "715710_2024-07-16", "708016_2024-04-29",
                    ]


session_id = example_session_ids[0] #session 4? mouse 4?

# loop through the directories to find the NWB file for the specific session 
for d in root.iterdir():
    nwb_path = d / f"{session_id}.nwb.zarr"
    if nwb_path.exists():
        print(nwb_path)
        break

/root/capsule/data/dynamicrouting_datacube/ecephys_759434_2025-02-04_12-27-22_nwb_2026-08-04_15-01-21/759434_2025-02-04.nwb.zarr


In [ ]:
session = pynwb.read_nwb(nwb_path)

In [ ]:
# Get trials dataframe and look at a few rows

trials = session.trials[:]
trials.tail(7)

In [ ]:
session.units[:].head()


In [ ]:
# QC thresholds from Shawn 

units_table = session.units.to_dataframe()


MAX_ISI_VIOLATIONS_RATIO = 0.5    # lower is better
MAX_AMPLITUDE_CUTOFF = 0.1        # lower is better
MIN_PRESENCE_RATIO = 0.95         # higher is better
MAX_ACTIVITY_DRIFT = 0.2 # lower is better, moderate threshold

good_units = units_table[
    (units_table.isi_violations_ratio < MAX_ISI_VIOLATIONS_RATIO) &
    (units_table.amplitude_cutoff < MAX_AMPLITUDE_CUTOFF) &
    (units_table.presence_ratio > MIN_PRESENCE_RATIO) & 
    (units_table.activity_drift < MAX_ACTIVITY_DRIFT)
]
print('%d of %d units passed quality control' % (len(good_units), len(units_table)))

In [ ]:
good_units.structure.value_counts()

In [ ]:

def find_nwb_path(root, session_id):
    for d in root.iterdir():
        nwb_path = d / f"{session_id}.nwb.zarr"
        if nwb_path.exists():
            return nwb_path
    return None


def apply_qc(units_table,
             max_isi_violations_ratio=0.5,
             max_amplitude_cutoff=0.1,
             min_presence_ratio=0.95,
             max_activity_drift=0.2):
    return units_table[
        (units_table.isi_violations_ratio < max_isi_violations_ratio) &
        (units_table.amplitude_cutoff < max_amplitude_cutoff) &
        (units_table.presence_ratio > min_presence_ratio) &
        (units_table.activity_drift < max_activity_drift)
    ]


def compute_region_counts_low_memory(root, session_ids, max_retries=2, retry_delay=2, **qc_kwargs):
    """
    Loads each session one at a time, applies QC, records region value_counts,
    then explicitly drops the units table / file handle before moving to the next session.
    Returns a sessions x regions DataFrame — never holds more than one session's raw data at once.
    """
    region_counts = {}
    qc_summary = {}
    failed_sessions = []

    for session_id in session_ids:
        nwb_path = find_nwb_path(root, session_id)
        if nwb_path is None:
            print(f"[{session_id}] no NWB file found, skipping")
            failed_sessions.append(session_id)
            continue

        for attempt in range(1, max_retries + 1):
            io = None
            try:
                io = pynwb.NWBHDF5IO(str(nwb_path), mode="r") if str(nwb_path).endswith('.nwb') \
                     else None
                # pynwb.read_nwb handles both hdf5/zarr internally, but doesn't expose the io
                # object for manual closing, so we call it directly and rely on `del` below.
                session = pynwb.read_nwb(nwb_path)

                units_table = session.units.to_dataframe()
                good_units = apply_qc(units_table, **qc_kwargs)

                region_counts[session_id] = good_units['structure'].value_counts()
                qc_summary[session_id] = (len(good_units), len(units_table))

                print(f"[{session_id}] {len(good_units)} of {len(units_table)} units passed QC")

                # explicitly drop references so they're eligible for garbage collection
                del units_table, good_units, session
                if io is not None:
                    io.close()
                    del io
                gc.collect()
                break

            except OSError as e:
                print(f"[{session_id}] attempt {attempt} failed: {e}")
                if io is not None:
                    try:
                        io.close()
                    except Exception:
                        pass
                if attempt < max_retries:
                    time.sleep(retry_delay)
                else:
                    failed_sessions.append(session_id)

    if failed_sessions:
        print(f"\n{len(failed_sessions)} session(s) failed to load: {failed_sessions}")

    counts_df = pd.DataFrame(region_counts).fillna(0).astype(int)
    summary_df = pd.DataFrame(qc_summary, index=['n_good', 'n_total']).T
    return counts_df, summary_df

In [ ]:
def apply_qc(units_table,
             max_isi_violations_ratio=0.5,
             max_amplitude_cutoff=0.1,
             min_presence_ratio=0.95,
             max_activity_drift=0.2):
    """Apply QC thresholds (from Shawn) to a units dataframe."""
    good_units = units_table[
        (units_table.isi_violations_ratio < max_isi_violations_ratio) &
        (units_table.amplitude_cutoff < max_amplitude_cutoff) &
        (units_table.presence_ratio > min_presence_ratio) &
        (units_table.activity_drift < max_activity_drift)
    ]
    return good_units

In [ ]:
def compute_region_counts(units_by_session, **qc_kwargs):
    """Apply QC per session, return a sessions x regions DataFrame of good-unit counts."""
    region_counts = {}
    for session_id, units_table in units_by_session.items():
        good_units = apply_qc(units_table, **qc_kwargs)
        region_counts[session_id] = good_units['structure'].value_counts()
        print(f"[{session_id}] {len(good_units)} of {len(units_table)} units passed QC")

    counts_df = pd.DataFrame(region_counts).fillna(0).astype(int)
    return counts_df

In [ ]:
def plot_region_counts(counts_df, title="Quality units per region per session"):
    fig, ax = plt.subplots(figsize=(max(10, counts_df.shape[1] * 1.2), 6))
    counts_df.T.plot(kind='bar', stacked=True, ax=ax)
    ax.set_xlabel("Session")
    ax.set_ylabel("Number of quality units")
    ax.set_title(title)
    ax.legend(title="Region", bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    return fig, ax

In [ ]:
counts_df, summary_df = compute_region_counts_low_memory(root, example_session_ids)

In [ ]:
counts_df.head()

In [ ]:
def plot_region_heatmap(counts_df, regions=None, title="Quality units per region per session",
                         cmap="viridis", annotate=True, log_scale=False):
    """
    counts_df: sessions x regions DataFrame (or regions x sessions — auto-oriented below)
    regions: optional list of region names to include (filters rows)
    """
    # counts_df comes out of compute_region_counts_low_memory as regions x sessions
    # (columns=session_id from the dict-of-Series construction), so orient explicitly:
    df = counts_df.copy()

    if regions is not None:
        df = df.loc[df.index.intersection(regions)]
        df = df.reindex(regions)  # preserve requested order, NaN for missing regions

    df = df.fillna(0)

    fig, ax = plt.subplots(figsize=(max(8, df.shape[1] * 0.9), max(4, df.shape[0] * 0.4)))

    plot_data = df.values.astype(float)
    norm = colors.LogNorm(vmin=max(plot_data.min(), 1), vmax=plot_data.max()) if log_scale else None

    im = ax.imshow(plot_data, aspect='auto', cmap=cmap, norm=norm)

    ax.set_xticks(range(df.shape[1]))
    ax.set_xticklabels(df.columns, rotation=45, ha='right')
    ax.set_yticks(range(df.shape[0]))
    ax.set_yticklabels(df.index)

    ax.set_xlabel("Session")
    ax.set_ylabel("Region")
    ax.set_title(title)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Number of quality units")

    if annotate:
        for i in range(df.shape[0]):
            for j in range(df.shape[1]):
                val = df.values[i, j]
                text_color = "white" if val < plot_data.max() * 0.5 else "black"
                ax.text(j, i, int(val), ha='center', va='center', fontsize=8, color=text_color)

    plt.tight_layout()
    plt.show()
    return fig, ax

In [ ]:
plot_region_heatmap(counts_df)

In [ ]:


def region_filter_widget(counts_df):
    all_regions = sorted(counts_df.index.tolist())

    region_select = widgets.SelectMultiple(
        options=all_regions,
        value=tuple(all_regions),  # default: all selected
        description='Regions:',
        layout=widgets.Layout(width='300px', height='250px')
    )

    select_all_btn = widgets.Button(description="Select all")
    clear_btn = widgets.Button(description="Clear")
    log_toggle = widgets.Checkbox(value=False, description='Log color scale')

    output = widgets.Output()

    def update_plot(*args):
        with output:
            output.clear_output(wait=True)
            selected = list(region_select.value)
            if not selected:
                print("Select at least one region.")
                return
            plot_region_heatmap(counts_df, regions=selected, log_scale=log_toggle.value)

    def select_all(_):
        region_select.value = tuple(all_regions)

    def clear_selection(_):
        region_select.value = ()

    select_all_btn.on_click(select_all)
    clear_btn.on_click(clear_selection)
    region_select.observe(update_plot, names='value')
    log_toggle.observe(update_plot, names='value')

    controls = widgets.HBox([select_all_btn, clear_btn, log_toggle])
    ui = widgets.VBox([controls, region_select])

    update_plot()  # initial render
    display(ui, output)

In [ ]:
region_filter_widget(counts_df)

In [ ]:
def qc_parameter_sweep(units_by_session,
                        isi_thresholds=(0.2, 0.3, 0.5),
                        amplitude_thresholds=(0.05, 0.1, 0.2),
                        presence_thresholds=(0.9, 0.95, 0.99),
                        drift_thresholds=(0.1, 0.2, 0.3)):
    """
    Sweep one QC parameter at a time. Returns {param_name: DataFrame(sessions x threshold_value)}
    of good-unit counts.
    """
    defaults = dict(
        max_isi_violations_ratio=0.5,
        max_amplitude_cutoff=0.1,
        min_presence_ratio=0.95,
        max_activity_drift=0.2,
    )

    sweep_configs = {
        'max_isi_violations_ratio': isi_thresholds,
        'max_amplitude_cutoff': amplitude_thresholds,
        'min_presence_ratio': presence_thresholds,
        'max_activity_drift': drift_thresholds,
    }

    sweep_results = {}
    for param_name, values in sweep_configs.items():
        results = {}
        for value in values:
            kwargs = defaults.copy()
            kwargs[param_name] = value
            counts_per_session = {
                session_id: len(apply_qc(units_table, **kwargs))
                for session_id, units_table in units_by_session.items()
            }
            results[value] = counts_per_session
        sweep_results[param_name] = pd.DataFrame(results)  # index=session_id, columns=threshold values

    return sweep_results


def plot_qc_sweep(sweep_results):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    for ax, (param_name, df) in zip(axes, sweep_results.items()):
        for session_id in df.index:
            ax.plot(df.columns, df.loc[session_id], marker='o', alpha=0.7, label=session_id)
        ax.set_title(param_name)
        ax.set_xlabel("threshold value")
        ax.set_ylabel("# good units")
    axes[0].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()